In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [3]:
df.shape

(569, 33)

In [4]:
df.drop(columns=['id','Unnamed: 32'] ,inplace = True )

In [5]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


# 1. Scratch

In [6]:
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler , LabelEncoder

In [7]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [8]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [10]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [11]:
X_train_tensor.shape , y_train_tensor.shape

(torch.Size([455, 30]), torch.Size([455]))

In [12]:
class MySimpleNN() :
    def __init__(self, X):
        self.weights = torch.rand(X.shape[1],1 ,dtype=torch.float64 ,requires_grad = True ) #tensor of shape(cols,1)
        self.bias = torch.zeros(1 , dtype = torch.float32 , requires_grad = True)
    def forward(self,X):
        z = torch.matmul(X,self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred
    def loss_func(self,y_pred ,y):
        #clamp pred to avoid log(0)
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred , epsilon , 1-epsilon)
        loss = -(y_train_tensor * torch.log(y_pred) +(1-y_train_tensor)*torch.log(1-y_pred)).mean()
        return loss

In [13]:
learning_rate = 0.1
epochs = 25

In [14]:
model = MySimpleNN(X_train_tensor)

for epoch in range(epochs):
    y_pred = model.forward(X_train_tensor)
    loss = model.loss_func(y_pred , y_train_tensor)
    loss.backward()
    with torch.no_grad() : #parameters update #to_grad -> to not track gradients 1.e to not make it part of computation graoh
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad

    model.weights.grad.zero_()
    model.bias.grad.zero_()
    print(f"Epoch:{epoch+1},Loss:{loss}")

Epoch:1,Loss:3.76269430706035
Epoch:2,Loss:3.6415489526140883
Epoch:3,Loss:3.512831731302847
Epoch:4,Loss:3.3782649028908964
Epoch:5,Loss:3.238866525625926
Epoch:6,Loss:3.0907515967376145
Epoch:7,Loss:2.9400136654609392
Epoch:8,Loss:2.784631167976545
Epoch:9,Loss:2.6223602723635073
Epoch:10,Loss:2.4572025018621106
Epoch:11,Loss:2.285596028370405
Epoch:12,Loss:2.1114464808246556
Epoch:13,Loss:1.9426471657080668
Epoch:14,Loss:1.7740279154497505
Epoch:15,Loss:1.6082738775228589
Epoch:16,Loss:1.4522528669550003
Epoch:17,Loss:1.3062971616464945
Epoch:18,Loss:1.1768427986087937
Epoch:19,Loss:1.0662026539483356
Epoch:20,Loss:0.975856983865021
Epoch:21,Loss:0.9058560384058778
Epoch:22,Loss:0.8544922541003273
Epoch:23,Loss:0.8185305350860285
Epoch:24,Loss:0.7940411542658531
Epoch:25,Loss:0.7773827231930523


In [15]:
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred = (y_pred > 0.9).float()
    accuracy = (y_pred == y_test_tensor).float().mean()
    print(f"Accuracy:{accuracy}")

Accuracy:0.6354262828826904


# 2. NN Module

In [16]:
import torch.nn as nn

class Model(nn.Module) : #This  is a neural network class that inherits form PyTorch's base class
    def __init__(self , num_features):
        super().__init__() #run the parent class's constructor 
        self.network = nn.Sequential(
            nn.Linear(num_features , 3), # (input_features_count , neuron count)
            nn.ReLU(),
            nn.Linear(3, 1),
            nn.Sigmoid()
        )
    def forward(self ,features):
        out = self.network(features)
        return out 

In [17]:
#create dataset
features = torch.rand(10,5)
model = Model(features.shape[1])
model

Model(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=3, bias=True)
    (1): ReLU()
    (2): Linear(in_features=3, out_features=1, bias=True)
    (3): Sigmoid()
  )
)

In [18]:
model(features) #can use model.forward() too , but model(features) is better because it does not bypass pytorch's internal system
#this is the output of the network , probabilities of each feature

tensor([[0.3072],
        [0.2601],
        [0.2765],
        [0.2813],
        [0.2886],
        [0.2424],
        [0.2800],
        [0.2697],
        [0.3447],
        [0.2696]], grad_fn=<SigmoidBackward0>)

In [19]:
#show model weights
for name , param in model.named_parameters():
    print(name , param)

network.0.weight Parameter containing:
tensor([[-0.1885,  0.3342,  0.0315, -0.1358,  0.2253],
        [ 0.0340,  0.3650,  0.4372,  0.1361, -0.2969],
        [-0.0274, -0.4294, -0.0473,  0.2760,  0.1361]], requires_grad=True)
network.0.bias Parameter containing:
tensor([ 0.4228,  0.1656, -0.1279], requires_grad=True)
network.2.weight Parameter containing:
tensor([[-0.4950, -0.4602,  0.4787]], requires_grad=True)
network.2.bias Parameter containing:
tensor([-0.4040], requires_grad=True)


In [20]:
for params in model.parameters():
    print(params)

Parameter containing:
tensor([[-0.1885,  0.3342,  0.0315, -0.1358,  0.2253],
        [ 0.0340,  0.3650,  0.4372,  0.1361, -0.2969],
        [-0.0274, -0.4294, -0.0473,  0.2760,  0.1361]], requires_grad=True)
Parameter containing:
tensor([ 0.4228,  0.1656, -0.1279], requires_grad=True)
Parameter containing:
tensor([[-0.4950, -0.4602,  0.4787]], requires_grad=True)
Parameter containing:
tensor([-0.4040], requires_grad=True)


In [21]:
model.network[0].weight

Parameter containing:
tensor([[-0.1885,  0.3342,  0.0315, -0.1358,  0.2253],
        [ 0.0340,  0.3650,  0.4372,  0.1361, -0.2969],
        [-0.0274, -0.4294, -0.0473,  0.2760,  0.1361]], requires_grad=True)

In [22]:
from torchinfo import summary

summary(model , input_size = (10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Sequential: 1-1                        [10, 1]                   --
│    └─Linear: 2-1                       [10, 3]                   18
│    └─ReLU: 2-2                         [10, 3]                   --
│    └─Linear: 2-3                       [10, 1]                   4
│    └─Sigmoid: 2-4                      [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

## Toy project 

In [23]:
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler , LabelEncoder
from sklearn.model_selection import train_test_split

In [24]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [25]:
df.shape


(569, 33)

In [26]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [27]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [28]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [29]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [30]:
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

In [31]:
import torch.nn as nn 

class MySimpleNN(nn.Module):

    def __init__(self , num_features):
        super().__init__()
        self.linear = nn.Linear(num_features , 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self , features):
        out = self.linear(features)
        out = self.sigmoid(out)
        return out

In [32]:
lr = 0.1
epochs = 25

In [33]:
loss_func = nn.BCELoss() #defining loss function

In [34]:
model = MySimpleNN(X_train_tensor.shape[1])
optimizer = torch.optim.SGD(model.parameters(),lr = lr)
 
for epoch in range(epochs) :
    y_pred = model(X_train_tensor)
    loss = loss_func(y_pred , y_train_tensor.view(-1,1)) #view(-1,1)-> (automatically calculates the size based on total elements.,no of cols )
    optimizer.zero_grad() #clear gradients
    loss.backward()
    optimizer.step() #params update
    print(f"Epoch:{epoch} , Loss:{loss}")

Epoch:0 , Loss:0.5862216353416443
Epoch:1 , Loss:0.48030149936676025
Epoch:2 , Loss:0.41757622361183167
Epoch:3 , Loss:0.37495267391204834
Epoch:4 , Loss:0.3435607850551605
Epoch:5 , Loss:0.3192076086997986
Epoch:6 , Loss:0.29960763454437256
Epoch:7 , Loss:0.28339341282844543
Epoch:8 , Loss:0.26969048380851746
Epoch:9 , Loss:0.25791069865226746
Epoch:10 , Loss:0.24764233827590942
Epoch:11 , Loss:0.2385876476764679
Epoch:12 , Loss:0.2305254191160202
Epoch:13 , Loss:0.22328738868236542
Epoch:14 , Loss:0.21674296259880066
Epoch:15 , Loss:0.21078908443450928
Epoch:16 , Loss:0.20534323155879974
Epoch:17 , Loss:0.2003382444381714
Epoch:18 , Loss:0.1957188993692398
Epoch:19 , Loss:0.19143939018249512
Epoch:20 , Loss:0.18746115267276764
Epoch:21 , Loss:0.18375153839588165
Epoch:22 , Loss:0.18028272688388824
Epoch:23 , Loss:0.17703072726726532
Epoch:24 , Loss:0.17397478222846985


In [35]:
with torch.no_grad():
    y_pred = model(X_test_tensor)
    y_pred = (y_pred>0.9).float()
    accuracy = (y_pred == y_test_tensor).float().mean()
    print(f"Accuracy:{accuracy}")

Accuracy:0.5811018943786621
